<a href="https://colab.research.google.com/github/Abhinaytechie/LangGraph/blob/main/Langraph_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langgraph langsmith

In [ ]:
!pip install langchain langchain-groq langchain-community

In [ ]:
from google.colab import userdata
groq_api_key=userdata.get('GROQ_API_KEY')
langsmith=userdata.get('LANGSMITH_API_KEY')


In [ ]:
import os
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"]=langsmith
os.environ["LANGCHAIN_PROJECT"]="langgraph-demo"

In [ ]:
from langchain_groq import ChatGroq

llm=ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct',groq_api_key=groq_api_key)
llm.invoke("Hellow")

AIMessage(content='Hello! How are you today? Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 12, 'total_tokens': 35, 'completion_time': 0.054072265, 'prompt_time': 0.000227655, 'queue_time': 0.300271636, 'total_time': 0.05429992}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_5d3e4e58e1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--17f0a7e0-95ec-4d52-b46c-30ae5b220540-0', usage_metadata={'input_tokens': 12, 'output_tokens': 23, 'total_tokens': 35})

Start Building ChatBot Using Langgraph


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
  messages:Annotated[list,add_messages]
graph_builder=StateGraph(State)

In [ ]:
def chatbot(state:State):
  return {"messages":llm.invoke(state["messages"])}

In [ ]:
graph_builder.add_node("Chatbot",chatbot)

In [ ]:
graph_builder.add_edge(START,"Chatbot")
graph_builder.add_edge("Chatbot",END)

In [ ]:
graph=graph_builder.compile()

In [ ]:
while True:
  user_input=input("User: ")
  if user_input.lower() in ["quit","q"]:
    print("Thank you for using our service")
    break
  for event in graph.stream({'messages':('user',user_input)}):
    print(event.values())
    print("--------------------------")
    for value in event.values():
      print(value)
      print("--------------------------")
      print(value['messages'])
      print("--------------------------")
      print("Assistant",value['messages'].content)

User: tell about onepiece
dict_values([{'messages': AIMessage(content='One Piece! It\'s a beloved Japanese manga and anime series created by Eiichiro Oda that has been entertaining fans worldwide for over two decades. Here\'s a brief overview:\n\n**Storyline**\n\nThe story follows Monkey D. Luffy, a young pirate with a stretchy body due to eating a magical fruit called the Gum-Gum Fruit (Gomu Gomu no Mi). Luffy\'s dream is to become the Pirate King, a legendary figure who has the power to unite the entire pirate world.\n\nLuffy sets sail on the Grand Line, a treacherous sea filled with powerful pirates, marine forces, and ancient secrets. Along the way, he meets a diverse group of allies, including Roronoa Zoro, a skilled swordsman; Usopp, a talented liar and marksman; Sanji, a charismatic cook; and Nami, a skilled navigator.\n\nTogether, they form the Straw Hat Pirates, a crew that aims to find the ultimate treasure known as "One Piece," which will grant the finder the title of Pirate